# Sign, Verify And Rotate Keys Without Fooling Yourself

Companion notebook for the [Signatures, Public Keys and PKI](https://codemia.io/courses/security_engineering/sec_signatures_public_keys_and_pki) lesson on Codemia.

**What you will build**: a working signature toolkit built on the `cryptography` library. You will compare Ed25519, ECDSA and
RSA on speed and size, sign a webhook payload and watch a single changed byte fail, then find the
bug that signatures do not catch, where the verifier checks one reading of a document and the
application acts on another. Finally you will simulate a key rotation and find the overlap window
that keeps every valid signature verifying.

## Setup

In [ ]:
%pip install -q cryptography numpy matplotlib

In [ ]:
import json
import time
import hashlib

import numpy as np
import matplotlib.pyplot as plt
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ed25519, ec, rsa, padding
from cryptography.exceptions import InvalidSignature

print('cryptography is ready')

## Concept recap

A signature lets anyone holding the public key check that a message came from the holder of the
private key and has not changed since. It does not hide the message, and it says nothing about what
the message means. Three things go wrong in practice, and this notebook builds each of them: choosing
an algorithm and parameters, signing bytes that are not the bytes the application acts on, and
rotating keys without a window in which valid signatures fail or old keys stay trusted forever.

## Three algorithms on one machine

In [ ]:
msg = b'x' * 512
keys = {
    'Ed25519': ed25519.Ed25519PrivateKey.generate(),
    'ECDSA P-256': ec.generate_private_key(ec.SECP256R1()),
    'RSA-2048 PSS': rsa.generate_private_key(public_exponent=65537, key_size=2048),
    'RSA-3072 PSS': rsa.generate_private_key(public_exponent=65537, key_size=3072),
}
pss = padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH)

def sign(name, k, m):
    if name.startswith('Ed'):
        return k.sign(m)
    if name.startswith('ECDSA'):
        return k.sign(m, ec.ECDSA(hashes.SHA256()))
    return k.sign(m, pss, hashes.SHA256())

def verify(name, pub, sig, m):
    if name.startswith('Ed'):
        pub.verify(sig, m)
    elif name.startswith('ECDSA'):
        pub.verify(sig, m, ec.ECDSA(hashes.SHA256()))
    else:
        pub.verify(sig, m, pss, hashes.SHA256())

def per_op(fn, n):
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n

print(f'{"algorithm":<14}{"sign us":>10}{"verify us":>11}{"sig bytes":>11}{"pubkey bytes":>14}')
for name, k in keys.items():
    pub = k.public_key()
    sig = sign(name, k, msg)
    s = per_op(lambda: sign(name, k, msg), 200)
    v = per_op(lambda: verify(name, pub, sig, msg), 200)
    pk = pub.public_bytes(serialization.Encoding.DER, serialization.PublicFormat.SubjectPublicKeyInfo)
    print(f'{name:<14}{s * 1e6:>10.0f}{v * 1e6:>11.0f}{len(sig):>11}{len(pk):>14}')

Ed25519 signs fastest by a wide margin, with 64 byte signatures and small keys, and it has no
parameters to get wrong and no per signature randomness to leak, which is why the lesson recommends
it for new designs. RSA verifies quickly but signs slowly and produces large signatures, which matters
for certificates and tokens sent on every request. ECDSA is in between and depends on a good random
number for every signature: reuse one and the private key falls out, which is how real consoles and
wallets were broken.

## A signed webhook, and what a signature catches

In [ ]:
signer = ed25519.Ed25519PrivateKey.generate()
pinned_public = signer.public_key()      # the receiver stores this ahead of time

body = b'{"event":"payout.created","amount":100,"currency":"USD","account":"acct_4412"}'
sig = signer.sign(body)

def receive(raw, signature, pub=pinned_public):
    try:
        pub.verify(signature, raw)
    except InvalidSignature:
        return 'rejected: bad signature'
    return 'accepted: ' + json.loads(raw)['event']

print(receive(body, sig))
print(receive(body.replace(b'100', b'900'), sig))
other = ed25519.Ed25519PrivateKey.generate()
print(receive(body, other.sign(body)), '(signed by a key we never pinned)')

## The bug a signature cannot catch: two readings of one document

In [ ]:
# The sender's system builds this body, and its own review tool shows the first amount.
crafted = b'{"event":"payout.created","amount":100,"currency":"USD","account":"acct_4412","amount":900}'
crafted_sig = signer.sign(crafted)

print(receive(crafted, crafted_sig))
print('the application acts on amount =', json.loads(crafted)['amount'])

The signature is valid, because it covers exactly these bytes, and the bytes contain two amounts.
Python's JSON parser keeps the last duplicate, other parsers keep the first, and some reject the
document. Whenever the component that approves or signs a document and the component that acts on it
can read it differently, an attacker who influences the content can make them disagree. The same gap
appears with Unicode normalisation, URL encoding and XML signature wrapping.

The fix has two parts: parse strictly, refusing duplicates and anything else ambiguous, and act on
exactly what was verified, never on a second parse or a re-serialised copy.

In [ ]:
def strict_loads(raw):
    def no_duplicates(pairs):
        seen = {}
        for k, v in pairs:
            if k in seen:
                raise ValueError(f'duplicate key {k!r}')
            seen[k] = v
        return seen
    return json.loads(raw, object_pairs_hook=no_duplicates)

def receive_strict(raw, signature):
    try:
        pinned_public.verify(signature, raw)
        doc = strict_loads(raw)
    except InvalidSignature:
        return 'rejected: bad signature'
    except ValueError as e:
        return f'rejected: {e}'
    return f'accepted: amount {doc["amount"]}'

print(receive_strict(body, sig))
print(receive_strict(crafted, crafted_sig))

# Re-serialising before verifying is the opposite mistake: the bytes change and honest messages fail.
reserialised = json.dumps(json.loads(body), indent=1).encode()
print('verify a re-serialised copy:', receive(reserialised, sig))

## Rotating a signing key

In [ ]:
# Day by day simulation. The signer switches to a new key on day SWITCH. Verifiers refresh their
# trusted key set from a published endpoint every REFRESH days, so each verifier learns about a new
# key some time after it is published. Signatures are checked up to MAX_AGE days after signing.
rng = np.random.default_rng(11)
VERIFIERS, REFRESH, MAX_AGE, SWITCH, DAYS = 500, 7, 30, 60, 150
phase = rng.integers(0, REFRESH, VERIFIERS)

def run(publish_lead, retire_after):
    publish = SWITCH - publish_lead        # day the new key appears in the published set
    retire = SWITCH + retire_after         # day the old key leaves the published set
    fails, stale = [], []
    for day in range(DAYS):
        last_refresh = day - ((day - phase) % REFRESH)
        knows_new = last_refresh >= publish
        knows_old = last_refresh < retire
        # signatures checked today were made any time in the last MAX_AGE days
        made = np.arange(max(0, day - MAX_AGE), day + 1)
        with_new = (made >= SWITCH).mean()
        fail = with_new * (~knows_new).mean() + (1 - with_new) * (~knows_old).mean()
        fails.append(fail)
        stale.append(knows_old.mean() if day > SWITCH + MAX_AGE else 0.0)
    return np.array(fails), np.array(stale)

plans = {'switch with no warning, retire at once': (0, 0),
         'publish 1 week early, retire at once': (7, 0),
         'publish 1 week early, retire after max age': (7, MAX_AGE),
         'publish 1 week early, never retire': (7, 10_000)}
fig, ax = plt.subplots(figsize=(8, 3.8))
for name, (lead, after) in plans.items():
    f, st = run(lead, after)
    ax.plot(f, label=name)
    print(f'{name:<44} worst day fails {f.max():>6.1%} of checks; '
          f'verifiers still trusting the old key on the last day: {st[-1]:.0%}')
ax.set_xlabel('day')
ax.set_ylabel('valid signatures rejected')
ax.legend(fontsize=7)
ax.set_title('Rotation needs both a lead and a tail')
plt.tight_layout()
plt.show()

Rotation fails in two directions. Switching without warning breaks every verifier that has not
refreshed yet, and retiring the old key at the switch breaks every signature made in the last
`MAX_AGE` days. The safe plan publishes the new key at least one refresh interval before using it and
keeps the old key until the last signature made with it has expired. The fourth line is the opposite
failure, invisible on the chart: an old key that is never retired stays trusted forever, so a leak
of it years later is still a working forgery key. Key identifiers in the signature, the `kid` used by
JWTs and JWKS endpoints, are what let both keys be trusted at once without trying each in turn.

## Exercises

1. Time signing and verification for 64 bytes and for 1 MB. Why does the message size barely matter
   for Ed25519, and what does that tell you about how it processes the message?
2. Extend `strict_loads` to reject numbers that would lose precision as floats, such as
   `9007199254740993`. Which real systems have been confused by that difference?
3. In the rotation simulation, set `REFRESH` to 30. What publish lead do you now need for zero
   failures, and what does that say about caching key sets for a long time?
4. Add a compromise on day 100: the old key leaks. Which plan limits forgery, and for how long are
   forged signatures accepted under each?

## What to take away

- Use Ed25519 for new signatures unless an ecosystem requires otherwise. It is fast, small and has
  no per signature randomness to get wrong.
- Pin the public key, or a trust anchor, ahead of time. A signature from a key you did not choose to
  trust proves nothing.
- A signature covers bytes, not meaning. Parse strictly, refuse ambiguity, and act on exactly the
  bytes you verified.
- Rotate with a lead and a tail: publish before use, retire after the last signature expires, and
  carry a key identifier so both keys can be trusted together.